In [1]:
import sys, os
path = os.path.abspath('../src')
sys.path.insert(0, path)

In [2]:
from gedih3.config import GH3_DEFAULT_DOWNLOAD_DIR, GH3_DEFAULT_SOC_DIR, GEDI_START_DATE
from gedih3.gedidriver import soc_file_tree, dask_h5_merged
from gedih3.gh3builder import h3_index_df, build_h3db_from_soc

/home/tiago/miniconda3/envs/gedih3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dask.distributed import Client
import dask.dataframe
import dask_geopandas
client = Client(processes=True, threads_per_worker=1, n_workers=2)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 2,Total memory: 15.49 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34617,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38663,Total threads: 1
Dashboard: http://127.0.0.1:39667/status,Memory: 7.74 GiB
Nanny: tcp://127.0.0.1:46391,


2025-10-04 23:30:28,801 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle bf198abf548d85d5ab2db5e34766bf0f initialized by task ('shuffle-transfer-bf198abf548d85d5ab2db5e34766bf0f', 1) executed on worker tcp://127.0.0.1:45789
2025-10-04 23:30:31,689 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle bf198abf548d85d5ab2db5e34766bf0f deactivated due to stimulus 'task-finished-1759635031.6808445'
2025-10-04 23:31:32,361 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 8cd9aa7bc5d0e88f4dc504923edf50e0 initialized by task ('shuffle-transfer-8cd9aa7bc5d0e88f4dc504923edf50e0', 1) executed on worker tcp://127.0.0.1:45789
2025-10-04 23:31:36,966 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 8cd9aa7bc5d0e88f4dc504923edf50e0 deactivated due to stimulus 'task-finished-1759635096.955136'
2025-10-04 23:32:15,471 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 80950a65989b3d8816ed69070a8b7ca1 initialized by task ('shuffle-transfer-80950a65989b3d8

In [9]:
ddf = dask_geopandas.read_parquet('../tmp', gather_spatial_partitions=False)

In [34]:
def calc(df):
    import h3pandas
    return df.h3.h3_to_parent(5).set_index('h3_05')

meta = calc(df)
meta

,shot_number,elev_lowestmode_l2a,lon_lowestmode_l2a,delta_time_l2a,rh_000_l2a,rh_001_l2a,rh_002_l2a,rh_003_l2a,rh_004_l2a,rh_005_l2a,...,wsci_xy_pi_upper_l4c,wsci_z_l4c,wsci_z_pi_lower_l4c,wsci_z_pi_upper_l4c,root_beam_l4c,root_file_l4c,datetime,geometry,h3_03,year
h3_05,,,,,,,,,,,,,,,,,,,,,
85804c6ffffffff,87320100300498317,57.338013,-50.832984,7.851114e+07,0.0,0.0,0.0,0.0,0.0,0.0,...,-9999.0,-9999.0,-9999.0,-9999.0,BEAM0001,GEDI04_C_2020179153021_O08732_03_T03677_02_001_01_V002.h5,2020-06-27 21:39:04.218766451,POINT (-50.83298 1.6027),83804cfffffffff,2020
85804c6ffffffff,87320100300498318,56.974083,-50.832686,7.851114e+07,0.0,0.0,0.0,0.0,0.0,0.0,...,-9999.0,-9999.0,-9999.0,-9999.0,BEAM0001,GEDI04_C_2020179153021_O08732_03_T03677_02_001_01_V002.h5,2020-06-27 21:39:04.227030993,POINT (-50.83269 1.60228),83804cfffffffff,2020
85804c6ffffffff,87320100300498319,57.206856,-50.832388,7.851114e+07,0.0,0.0,0.0,0.0,0.0,0.0,...,-9999.0,-9999.0,-9999.0,-9999.0,BEAM0001,GEDI04_C_2020179153021_O08732_03_T03677_02_001_01_V002.h5,2020-06-27 21:39:04.235295057,POINT (-50.83239 1.60186),83804cfffffffff,2020
85804c6ffffffff,87320100300498320,57.221401,-50.832089,7.851114e+07,0.0,0.0,0.0,0.0,0.0,0.0,...,-9999.0,-9999.0,-9999.0,-9999.0,BEAM0001,GEDI04_C_2020179153021_O08732_03_T03677_02_001_01_V002.h5,2020-06-27 21:39:04.243558884,POINT (-50.83209 1.60145),83804cfffffffff,2020
85804c6ffffffff,87320100300498321,57.393181,-50.831791,7.851114e+07,0.0,0.0,0.0,0.0,0.0,0.0,...,-9999.0,-9999.0,-9999.0,-9999.0,BEAM0001,GEDI04_C_2020179153021_O08732_03_T03677_02_001_01_V002.h5,2020-06-27 21:39:04.251822948,POINT (-50.83179 1.60103),83804cfffffffff,2020


In [ ]:
a = ddf.query("wsci_l4c > 0").map_partitions(calc, meta=meta).groupby('h3_03', observed=True)[['wsci_l4c','agbd_l4a']].apply(lambda x: x.groupby(level=0).agg('mean'))
a = a.map_partitions(lambda x: x.droplevel(0))

/tmp/ipykernel_835595/1933781529.py:1: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  a = ddf.query("wsci_l4c > 0").map_partitions(calc, meta=meta).groupby('h3_03', observed=True)[['wsci_l4c','agbd_l4a']].apply(lambda x: x.groupby(level=0).agg('mean'))


In [69]:
a.compute()

,wsci_l4c,agbd_l4a
h3_05,,
85804103fffffff,7.801361,-4579.232422
85804107fffffff,8.467340,-328.460724
8580410bfffffff,7.000627,1.314341
8580410ffffffff,7.969027,-5810.027832
85804113fffffff,10.157640,55.032093
...,...,...
85804ec7fffffff,8.622376,-848.565979
85804ecbfffffff,8.066348,-9999.000000
85804ecffffffff,8.746167,-9999.000000
